# 📘 Document Question Answering System (RAG) using  Gemini API

---

## Step 1: Setup - imports and Gemini config

In [1]:
!pip install google-generativeai PyPDF2 python-docx numpy -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 7.7 MB/s eta 0:00:00


In [2]:
import os
import numpy as np
import google.generativeai as genai
from PyPDF2 import PdfReader
from docx import Document

api_key = "my api key"
genai.configure(api_key=api_key)

embed_model = "models/gemini-embedding-001"
chat_model = genai.GenerativeModel("gemini-flash-latest")

print("setup done")

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


setup done


## Step 2: Document ingestion function

In [3]:
def load_document(path):
    if path.endswith(".pdf"):
        reader = PdfReader(path)
        text = "".join(page.extract_text() or "" for page in reader.pages)
    elif path.endswith(".docx"):
        doc = Document(path)
        text = "\n".join(p.text for p in doc.paragraphs)
    elif path.endswith(".txt"):
        text = open(path, "r", encoding="utf-8").read()
    else:
        print("file type not supported")
        text = ""
    return text

## Step 3: Chunking function

In [4]:
def make_chunks(text, chunk_size=500, overlap=50):
    chunks = []
    start = 0
    while start < len(text):
        chunks.append(text[start:start + chunk_size])
        start += chunk_size - overlap
    return chunks

## Step 4: Embedding function (Gemini)

In [5]:
def get_embedding(text):
    return genai.embed_content(model=embed_model, content=text)["embedding"]

## Step 5: Vector similarity and retrieval

In [6]:
def cosine_sim(a, b):
    a, b = np.array(a), np.array(b)
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

def retrieve_top_chunks(question, chunks, chunk_embeddings, top_k=3):
    q_emb = get_embedding(question)
    scores = [cosine_sim(q_emb, emb) for emb in chunk_embeddings]
    ranked = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:top_k]
    return [chunks[i] for i in ranked], [scores[i] for i in ranked]

## Step 6: Answer generation (Gemini)

In [7]:
def generate_answer(question, top_chunks, top_scores, threshold=0.5):
    if max(top_scores) < threshold:
        return "This information is not present in the uploaded document."
    context = "\n\n".join(top_chunks)
    prompt = (f"Answer the question using only the context below. "
              f"If the answer is not in the context, say it is not found in the document.\n\n"
              f"Context:\n{context}\n\nQuestion: {question}")
    return chat_model.generate_content(prompt).text

## Step 7: Choose or Upload a Document

In [8]:
from google.colab import files

uploaded = files.upload()

Saving RAG_University_Dataset.pdf to RAG_University_Dataset.pdf


In [9]:
import os

# Name of the uploaded file
default_path = "sample_docs/sample.txt"
file_path = "RAG_University_Dataset.pdf"

if not os.path.exists(file_path):
    print("File not found, falling back to sample document")
    file_path = default_path

print("Using file:", file_path)

Using file: RAG_University_Dataset.pdf


## Step 8: Run ingestion, chunking, and embedding

In [10]:
import os

print(os.getenv("GOOGLE_API_KEY"))

None


In [14]:
import google.generativeai as genai

api_key = "YOUR_API_KEY"

genai.configure(api_key=api_key)

for model in genai.list_models():
    print(model.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-3.5-flash
models/gemini-omni-flash-preview
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/gemini-3.1-flash-tts-preview
models/gemini-robotics-er-1.5-preview
mod

In [15]:
raw_text = load_document(file_path)
chunks = make_chunks(raw_text)
chunk_embeddings = [get_embedding(c) for c in chunks]

print("total characters loaded:", len(raw_text))
print("total chunks made:", len(chunks))
print("embedding size:", len(chunk_embeddings[0]))

total characters loaded: 2260
total chunks made: 6
embedding size: 3072


## Step 9: Testing it out with a few questions

In [16]:
import time
test_questions = [
    "What is the admission process?",
    "What is the eligibility criteria for undergraduate admission?",
    "What is the annual B.Tech tuition fee?",
    "How much is the hostel fee per year?",
    "Who can apply for the merit scholarship?",
    "How many books can a student borrow from the library?",
    "What is the late fine for returning library books?"
]

for q in test_questions:
    top_chunks, top_scores = retrieve_top_chunks(q, chunks, chunk_embeddings)
    answer = generate_answer(q, top_chunks, top_scores)
    print("Q:", q)
    print("top similarity score:", round(max(top_scores), 3))
    print("A:", answer)
    print("---")
    time.sleep(13)

Q: What is the admission process?
top similarity score: 0.624
A: Based on the provided document, the admission process is as follows: 

Students can apply online by filling out the admission form, uploading the required documents, and paying the application fee.
---
Q: What is the eligibility criteria for undergraduate admission?
top similarity score: 0.64
A: Based on the provided context, undergraduate applicants must have completed 12th grade with at least 60% marks from a recognized board.
---
Q: What is the annual B.Tech tuition fee?
top similarity score: 0.666
A: Based on the provided document, the annual tuition fee for B.Tech is Rs. 1,20,000.
---
Q: How much is the hostel fee per year?
top similarity score: 0.709
A: Based on the provided context, the hostel fee is Rs. 60,000 per year.
---
Q: Who can apply for the merit scholarship?
top similarity score: 0.586
A: Based on the provided document, merit scholarships are available for students scoring above 90%.
---
Q: How many books

## Step 10: System metrics report

In [17]:
print("chunk size:", 500, "| overlap:", 50, "| total chunks:", len(chunks))
print("embedding model:", embed_model, "| dimension:", len(chunk_embeddings[0]))
print("llm:", "gemini-flash-latest", "| vector store: in-memory list + cosine similarity")
print("retrieval top_k:", 3)

chunk size: 500 | overlap: 50 | total chunks: 6
embedding model: models/gemini-embedding-001 | dimension: 3072
llm: gemini-flash-latest | vector store: in-memory list + cosine similarity
retrieval top_k: 3


## Step 11: Ask your own question

In [18]:
question = "How can I get a job in Celebal Technologies?"

top_chunks, top_scores = retrieve_top_chunks(question, chunks, chunk_embeddings)
answer = generate_answer(question, top_chunks, top_scores)

print("Question:", question)
print("Answer:", answer)

Question: How can I get a job in Celebal Technologies?
Answer: it is not found in the document.
